# OOD benchmark — Whisper-large-v3 yo fine-tune vs baseline

Tests the v4 fine-tune on **out-of-distribution audio from a public corpus that was never in training**. This is the thesis-grade generalization claim — paired with the in-distribution holdout from `Whisper_test.ipynb`, you get both axes of the model's quality.

**Default OOD corpus**: `openslr/openslr` config `SLR86` (Lagos University Yorùbá corpus, ~4h read speech). No gating, different author from your training data — the safest OOD claim for the thesis.

**Alternatives**: FLEURS yo_ng test split (Step 4 block B), or any HF audio dataset by editing block C. **Mozilla Common Voice is intentionally not the default** — newer CV versions have Hub streaming issues, and the project's preference is to avoid the Mozilla pipeline.

**What you get**: a paired `baseline vs v4` table with YASR-Bench metrics (Y-WER, Y-WER-perm, Y-CER, Y-CER-perm, Δ), plus a JSON payload saved next to the training run on Drive — `RUN_DIR/ood_<dataset>_paired.json`.

**Runtime**: any GPU with ≥6 GB VRAM. Colab T4 is fine. ~5–10 min per model at N_OOD=100; bump to 500 for the final thesis number.

## Step 1 — Auth (HF + Drive)

The **only cell that prompts you**. Authorize Drive in the popup, walk away while the rest runs.

In [ ]:
import os
from pathlib import Path

# --- Drive first ---
try:
    from google.colab import drive
    try:
        drive.mount("/content/drive", force_remount=False)
    except ValueError:
        drive.mount("/content/drive", force_remount=True)
    DRIVE_TRAINING = Path("/content/drive/MyDrive/yoruba-pipeline-logs/training")
    print(f"Drive mounted → {DRIVE_TRAINING}")
except Exception as e:
    DRIVE_TRAINING = Path("./yoruba-pipeline-logs/training")
    print(f"Drive unavailable ({type(e).__name__}); using local fallback → {DRIVE_TRAINING}")

# --- HF auth ---
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("HF auth: ok")
else:
    print("HF auth: skipped — Common Voice will fail (gated dataset). Set HF_TOKEN secret.")

## Step 2 — Install dependencies

In [ ]:
%%capture
!pip install -q "transformers>=4.45" "datasets>=3.0" "huggingface_hub>=0.24" \
    librosa soundfile evaluate jiwer torchcodec accelerate

## Step 3 — Config

Single source of truth. `RUN_ID = None` auto-picks the most recent training run on Drive with a merged checkpoint.

In [ ]:
import torch

DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE           = torch.float16 if DEVICE == "cuda" else torch.float32
TASK            = "transcribe"
BASELINE_ID     = "openai/whisper-large-v3"

N_OOD               = 100        # bump to 200–500 for the final thesis number
NORMALIZER_VERSION  = "v1"

# Which training run to test. None = pick the most recent one with merged_16bit.
RUN_ID = None

print(f"device  = {DEVICE}  dtype = {DTYPE}")
print(f"N_OOD   = {N_OOD}")

## Step 4 — Pick the OOD corpus

Default: Mozilla Common Voice 17 yo. Uncomment the OpenSLR86 block (and comment out CV) to switch.

In [ ]:
# A) OpenSLR86 — Lagos University Yorùbá corpus. No gating, ~4h read speech.
#    Different author from training data → safest OOD claim for the thesis.
OOD_DATASET  = "openslr/openslr"
OOD_CONFIG   = "SLR86"
OOD_SPLIT    = "train"
OOD_TEXT_COL = "transcription"

# B) FLEURS yo_ng TEST split — Google's multi-speaker read speech.
#    Note: training used google/fleurs yo_ng TRAIN. The test split is held out
#    from training, so it's OOD against the trained model, but a strict reviewer
#    might say "same dataset, different split" rather than fully OOD. Use this
#    if OpenSLR86 doesn't stream — it's the project's standard benchmark anyway.
# OOD_DATASET  = "google/fleurs"
# OOD_CONFIG   = "yo_ng"
# OOD_SPLIT    = "test"
# OOD_TEXT_COL = "raw_transcription"   # 'transcription' is also present (normalised)

# C) Bring your own — any HF audio dataset with audio + transcript columns
# OOD_DATASET  = "<owner>/<dataset>"
# OOD_CONFIG   = None              # or e.g. "yo"
# OOD_SPLIT    = "test"
# OOD_TEXT_COL = "text"            # whatever column holds the reference text

print(f"OOD = {OOD_DATASET}:{OOD_CONFIG}:{OOD_SPLIT}")

## Step 5 — Find the v4 model on Drive

Auto-discovers `merged_16bit/` under `MyDrive/yoruba-pipeline-logs/training/`. Override `RUN_ID` in Step 3 to pin a specific run.

Also loads the training run's `config.json` so you have full provenance of what's being tested.

In [ ]:
import json

if not DRIVE_TRAINING.exists():
    raise FileNotFoundError(
        f"No training runs at {DRIVE_TRAINING}. Run Whisper_v4.ipynb first."
    )

if RUN_ID is None:
    candidates = sorted(
        [p for p in DRIVE_TRAINING.iterdir() if (p / "merged_16bit").exists()],
        reverse=True,
    )
    if not candidates:
        raise FileNotFoundError(
            f"No run with merged_16bit/ in {DRIVE_TRAINING}.\n"
            "Re-run Whisper_v4.ipynb to completion with SAVE_MERGED_DRIVE=True."
        )
    RUN_DIR = candidates[0]
    RUN_ID  = RUN_DIR.name
else:
    RUN_DIR = DRIVE_TRAINING / RUN_ID
    if not (RUN_DIR / "merged_16bit").exists():
        raise FileNotFoundError(f"No merged_16bit/ in {RUN_DIR}")

V4_PATH = str(RUN_DIR / "merged_16bit")

run_cfg = {}
if (RUN_DIR / "config.json").exists():
    with (RUN_DIR / "config.json").open() as f:
        run_cfg = json.load(f)

print(f"Run     : {RUN_ID}")
print(f"Path    : {V4_PATH}")
if run_cfg:
    print(f"Trained on : {run_cfg.get('dataset', {}).get('repo')}")
    print(f"Splits     : holdout={run_cfg['splits']['holdout']}  "
          f"train={run_cfg['splits']['train']}  eval={run_cfg['splits']['eval']}")
    print(f"Epochs     : up to {run_cfg['trainer']['num_epochs']}")

# Sanity: refuse to run if OOD corpus overlaps training repo
training_repo = run_cfg.get("dataset", {}).get("repo", "")
if training_repo and OOD_DATASET.lower() in training_repo.lower():
    raise RuntimeError(
        f"OOD_DATASET={OOD_DATASET} overlaps with training repo {training_repo}.\n"
        "Pick a different OOD corpus in Step 4."
    )

## Step 6 — Cache OOD samples

Streams `N_OOD` clips from the OOD corpus into memory **once**, so both models see identical audio. Skips clips with empty references.

In [ ]:
import itertools
from datasets import load_dataset, Audio

print(f"Streaming {OOD_DATASET} ({OOD_CONFIG}, {OOD_SPLIT})…")
try:
    if OOD_CONFIG is None:
        ood_stream = load_dataset(OOD_DATASET, split=OOD_SPLIT, streaming=True)
    else:
        ood_stream = load_dataset(OOD_DATASET, OOD_CONFIG, split=OOD_SPLIT, streaming=True)
    ood_stream = ood_stream.cast_column("audio", Audio(sampling_rate=16000))
except Exception as e:
    raise RuntimeError(
        f"Couldn't load {OOD_DATASET}: {type(e).__name__}: {e}\n\n"
        f"Common causes:\n"
        f"  - Wrong config name. For OpenSLR86 it's 'SLR86' (uppercase, no spaces).\n"
        f"  - Wrong split. OpenSLR86 only has 'train'. FLEURS has 'train'/'validation'/'test'.\n"
        f"  - Wrong text column. Peek with:\n"
        f"      next(iter(load_dataset({OOD_DATASET!r}, {OOD_CONFIG!r}, split={OOD_SPLIT!r}, streaming=True)))\n"
        f"  - HF_TOKEN missing: re-run Step 1.\n"
        f"\nFallbacks to try (Step 4):\n"
        f"  - google/fleurs:yo_ng:test  (held-out from training, well-tested streaming)\n"
        f"  - openslr/openslr:SLR86:train  (Lagos Yorùbá corpus)\n"
    ) from e

OOD_SAMPLES = []
for s in itertools.islice(ood_stream, N_OOD * 2):
    ref = s.get(OOD_TEXT_COL) or s.get("text") or s.get("transcription") or s.get("raw_transcription") or ""
    if ref.strip() and len(OOD_SAMPLES) < N_OOD:
        OOD_SAMPLES.append({"audio": s["audio"]["array"], "ref": ref})

OOD_REFS = [s["ref"] for s in OOD_SAMPLES]
print(f"  cached {len(OOD_SAMPLES)} samples")

## Step 7 — Metrics + helpers (YASR-Bench)

Same `normalize` / `strip_diacritics` / `yasr_bench` definitions as `Whisper_test.ipynb`. If you ever change these, bump `NORMALIZER_VERSION` in Step 3 so historical JSON payloads stay distinguishable.

In [ ]:
import re
import unicodedata
import evaluate

_PUNCT_RE = re.compile(r"[^\w\s]", flags=re.UNICODE)
_WS_RE    = re.compile(r"\s+")

def normalize(s: str) -> str:
    s = s.lower()
    s = _PUNCT_RE.sub(" ", s)
    return _WS_RE.sub(" ", s).strip()

def strip_diacritics(s: str) -> str:
    s = unicodedata.normalize("NFD", s)
    s = "".join(ch for ch in s if unicodedata.category(ch) != "Mn")
    return unicodedata.normalize("NFC", s)

wer_m = evaluate.load("wer")
cer_m = evaluate.load("cer")

def yasr_bench(refs, hyps):
    refs_n = [normalize(r) for r in refs]
    hyps_n = [normalize(h) for h in hyps]
    refs_p = [strip_diacritics(r) for r in refs_n]
    hyps_p = [strip_diacritics(h) for h in hyps_n]
    return {
        "Y-WER":      100 * wer_m.compute(predictions=hyps_n, references=refs_n),
        "Y-WER-perm": 100 * wer_m.compute(predictions=hyps_p, references=refs_p),
        "Y-CER":      100 * cer_m.compute(predictions=hyps_n, references=refs_n),
        "Y-CER-perm": 100 * cer_m.compute(predictions=hyps_p, references=refs_p),
    }

print("metrics loaded")

## Step 7.5 — Overlap check (OOD ↔ training)\n\nVets the chosen OOD corpus against the *actual* training dataset recorded in `run_cfg`. Catches the same-author-different-shard footgun (e.g. `Hidi-agili/yoruba_male_dataset` vs `Hidi-agili/yoruba_tts_dataset`).\n\n**What it checks**: text-level overlap between OOD transcripts and training transcripts, after normalization. Same-author corpora sometimes share clip transcripts verbatim — the audio is reused with the same labels.\n\n**Limits**:\n- Only catches **exact normalized-text matches**. Audio clips re-used with different formatting/diacritization slip through. For thesis-grade certainty, run an audio fingerprint check too — heavier, deferred to `scripts/` if needed.\n- For the Common Voice / OpenSLR86 defaults this should print `✅ No overlap` instantly. For `Hidi-agili/yoruba_male_dataset` we expect a non-trivial overlap percentage.\n\n**Behaviour**:\n- 0% overlap → ✅ continues.\n- 0% < overlap < 10% → ⚠ warns, lists first few overlapping indices, continues.\n- ≥ 10% overlap → **aborts the cell** with `RuntimeError`. Pick a different OOD corpus.

In [ ]:
from datasets import load_dataset
from tqdm.auto import tqdm

# Thresholds (tweak per-thesis tolerance)
OVERLAP_ABORT_PCT = 10.0    # ≥ this → RuntimeError, refuse to evaluate
OVERLAP_WARN_PCT  = 1.0     # ≥ this (but < abort) → loud warning

training_repo  = run_cfg.get("dataset", {}).get("repo")
training_split = run_cfg.get("dataset", {}).get("split", "train")

if not training_repo:
    print("⚠  No training dataset recorded in config.json — skipping overlap check.")
    print("   You can't make a thesis OOD claim without this. Pin RUN_ID to a run with config.json.")
else:
    print(f"Checking text overlap")
    print(f"  OOD      : {OOD_DATASET}:{OOD_CONFIG}:{OOD_SPLIT}  (N={len(OOD_REFS)})")
    print(f"  training : {training_repo}:{training_split}\n")

    # Stream the training corpus once, peek the first row to find the text column
    train_iter = iter(load_dataset(training_repo, split=training_split, streaming=True))
    first = next(train_iter)
    train_text_col = next(
        (c for c in ("text", "sentence", "transcription", "raw_transcription") if c in first),
        None,
    )
    if train_text_col is None:
        raise RuntimeError(
            f"Could not identify the text column in {training_repo}. "
            f"Available columns: {list(first.keys())}"
        )

    # Build the set of normalized training transcripts (streaming, no full download)
    train_norm_texts: set[str] = {normalize(first.get(train_text_col, ""))}
    for row in tqdm(train_iter, desc=f"indexing {training_repo}", total=None):
        t = row.get(train_text_col, "")
        if t and t.strip():
            train_norm_texts.add(normalize(t))
    train_norm_texts.discard("")
    print(f"\n  indexed {len(train_norm_texts):,} unique normalized training transcripts")

    # Compare OOD against the training set
    ood_norm = [normalize(r) for r in OOD_REFS]
    overlap_idx = [i for i, t in enumerate(ood_norm) if t in train_norm_texts]
    overlap_pct = 100 * len(overlap_idx) / len(OOD_REFS)

    print(f"\n  Overlap  : {len(overlap_idx)} / {len(OOD_REFS)}  ({overlap_pct:.1f}%)")

    if overlap_pct >= OVERLAP_ABORT_PCT:
        for i in overlap_idx[:5]:
            print(f"    overlapping[{i}]: {ood_norm[i][:100]}")
        raise RuntimeError(
            f"\nOOD overlap {overlap_pct:.1f}% ≥ {OVERLAP_ABORT_PCT}% — refusing to evaluate.\n"
            f"This corpus shares transcripts with training data; results would not be a\n"
            f"defensible OOD claim. Pick a different OOD dataset in Step 4."
        )
    elif overlap_pct >= OVERLAP_WARN_PCT:
        print(f"\n  ⚠  Non-zero overlap. Continuing, but document this in the thesis methodology.")
        print(f"     First overlapping OOD indices:")
        for i in overlap_idx[:5]:
            print(f"       [{i}] {ood_norm[i][:100]}")
    else:
        print(f"\n  ✅ Below {OVERLAP_WARN_PCT}% — corpus is safe for an OOD claim on text-level grounds.")
        print(f"     (Audio fingerprint overlap not verified; trivial for different-author corpora.)")

## Step 8 — Eval both models

Loads baseline → transcribes → unloads. Then loads v4 → transcribes → unloads. Sequential so even a T4 can run the full comparison.

In [ ]:
import gc
from tqdm.auto import tqdm
from transformers import WhisperProcessor, WhisperForConditionalGeneration

def eval_model(model_id_or_path: str, short_name: str):
    print(f"\n  loading {short_name} from {model_id_or_path}")
    proc = WhisperProcessor.from_pretrained(model_id_or_path)
    mdl  = WhisperForConditionalGeneration.from_pretrained(
        model_id_or_path, torch_dtype=DTYPE,
    ).to(DEVICE).eval()
    mdl.generation_config.language = "<|yo|>"
    mdl.generation_config.task = TASK
    mdl.generation_config.forced_decoder_ids = None

    hyps = []
    with torch.inference_mode():
        for s in tqdm(OOD_SAMPLES, desc=short_name):
            feats = proc.feature_extractor(
                s["audio"], sampling_rate=16000, return_tensors="pt",
            ).input_features.to(DEVICE, dtype=DTYPE)
            ids = mdl.generate(feats, language="<|yo|>", task=TASK,
                               max_new_tokens=256, num_beams=1)
            hyps.append(proc.tokenizer.batch_decode(ids, skip_special_tokens=True)[0])
    del mdl, proc
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return hyps

hyps_baseline = eval_model(BASELINE_ID, "baseline")
hyps_v4       = eval_model(V4_PATH,     "v4")

## Step 9 — Score + paired Δ table

In [ ]:
m_b  = yasr_bench(OOD_REFS, hyps_baseline)
m_v4 = yasr_bench(OOD_REFS, hyps_v4)
deltas = {k: m_b[k] - m_v4[k] for k in m_b}
# Δ > 0 means baseline is worse → v4 wins by Δ pp

print(f"\n=== OOD: {OOD_DATASET}:{OOD_CONFIG}:{OOD_SPLIT}  N={len(OOD_REFS)} ===\n")
print(f"{'metric':<14}  {'baseline':>10}  {'v4 (' + RUN_ID[:10] + ')':>20}  {'Δ (baseline−v4)':>17}")
print("-" * 70)
for k in ("Y-WER", "Y-WER-perm", "Y-CER", "Y-CER-perm"):
    print(f"{k:<14}  {m_b[k]:>9.2f}%  {m_v4[k]:>19.2f}%  {deltas[k]:>+16.2f}")
print(f"{'diac gap':<14}  "
      f"{m_b['Y-WER']-m_b['Y-WER-perm']:>9.2f}   "
      f"{m_v4['Y-WER']-m_v4['Y-WER-perm']:>19.2f}   "
      f"{(m_b['Y-WER']-m_b['Y-WER-perm']) - (m_v4['Y-WER']-m_v4['Y-WER-perm']):>+16.2f}")

## Step 10 — Persist results to Drive

In [ ]:
import datetime as _dt

payload = {
    "timestamp_utc":      _dt.datetime.now(_dt.timezone.utc).isoformat(),
    "run_id":             RUN_ID,
    "dataset":            f"{OOD_DATASET}:{OOD_CONFIG}:{OOD_SPLIT}",
    "n_eval":             len(OOD_REFS),
    "normalizer_version": NORMALIZER_VERSION,
    "baseline_model":     BASELINE_ID,
    "fine_tune_path":     V4_PATH,
    "metrics": {
        BASELINE_ID: m_b,
        RUN_ID:      m_v4,
    },
    "deltas_baseline_minus_v4": deltas,
    "refs":          OOD_REFS,
    "hyps_baseline": hyps_baseline,
    "hyps_v4":       hyps_v4,
}

slug = OOD_DATASET.split("/")[-1].replace("_", "-")
out_path = RUN_DIR / f"ood_{slug}_paired.json"
with out_path.open("w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)
print(f"wrote {out_path}")

## Step 11 — Side-by-side samples

5 three-way comparisons (REF / baseline / v4) on normalized text. Use these to eyeball *what kind* of errors the fine-tune still makes.

In [ ]:
print(f"\n--- 5 OOD samples (normalized) ---")
for i in range(min(5, len(OOD_REFS))):
    print(f"REF      : {normalize(OOD_REFS[i])}")
    print(f"baseline : {normalize(hyps_baseline[i])}")
    print(f"v4       : {normalize(hyps_v4[i])}\n")